In [ ]:
import scanpy as sc
import numpy as np
from collections import Counter

# adata = sc.read_h5ad("./data/mouse_organogenesis/SeqFISH_Embryo1_adata_SIRV_velocity.h5ad")
adata = sc.read_h5ad("./data/mouse_organogenesis/SeqFISH_Embryo2_adata_SIRV_velocity.h5ad")

print(adata)
print("Layers:", adata.layers.keys())
print("obsm:", adata.obsm.keys())
Counter(adata.obs["celltype_mapped_refined"])

In [ ]:
from scipy.sparse import issparse

X_raw = adata.X
if issparse(X_raw):
    X_raw = X_raw.toarray()

V_raw = adata.layers["velocity"]
if issparse(V_raw):
    V_raw = V_raw.toarray()

X_emb = adata.obsm["X_xy_loc"]
X_emb = X_emb.astype(float)

In [ ]:
from scripts.VectorFieldEmbedder import *

emb = VectorFieldEmbedder(
    X_raw, V_raw,
    dist_method="phase",
    dof=50,
    X_emb=X_emb,    # ADDED
    alpha=0.75,
    knn_k=30,
    max_tps_points=4000
)
emb.initialize_embedding(seed=42)

In [ ]:
simplify_map = {
    "Forebrain/Midbrain/Hindbrain": "Neural",
    "Spinal cord": "Neural",
    "Neural crest": "Neural",
    "NMP": "Neural",

    "Cranial mesoderm": "Mesoderm (Somitic)",
    "Dermomyotome": "Mesoderm (Somitic)",
    "Sclerotome": "Mesoderm (Somitic)",
    "Anterior somitic tissues": "Mesoderm (Somitic)",
    "Presomitic mesoderm": "Mesoderm (Somitic)",

    "Cardiomyocytes": "Mesoderm (LPM/Heart)",
    "Splanchnic mesoderm": "Mesoderm (LPM/Heart)",
    "Lateral plate mesoderm": "Mesoderm (LPM/Heart)",
    "Mixed mesenchymal mesoderm": "Mesoderm (LPM/Heart)",
    "Intermediate mesoderm": "Mesoderm (LPM/Heart)",

    "Gut tube": "Endoderm",
    "Definitive endoderm": "Endoderm",
    "ExE endoderm": "Endoderm",

    "Endothelium": "Endothelium/Blood",
    "Haematoendothelial progenitors": "Endothelium/Blood",
    "Blood progenitors": "Endothelium/Blood",
    "Erythroid": "Endothelium/Blood",

    "Surface ectoderm": "Surface ectoderm",

    "Allantois": "Allantois",
}

# Apply it
adata.obs["celltype_simplified"] = adata.obs["celltype_mapped_refined"].map(simplify_map)

Counter(adata.obs["celltype_simplified"])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from scripts.plotting import compute_velocity_on_grid


# ===============================================================
# Helper functions
# ===============================================================

def points_inside_mask(X_emb, seeds, k=8, radius_scale=1.2):
    nn = NearestNeighbors(n_neighbors=k).fit(X_emb)
    r = np.median(nn.kneighbors(X_emb, n_neighbors=k)[0][:, -1]) * radius_scale
    neigh_idx = nn.radius_neighbors(seeds, radius=r, return_distance=False)
    return np.array([len(ix) > 0 for ix in neigh_idx])


def add_manual_arrows(ax, coords, X_emb, V_pred,
                      color="k", scale=2.0, width=0.004):
    """Add optional manual arrows at chosen coords."""
    if len(coords) == 0:
        return ax

    nn = NearestNeighbors(n_neighbors=1).fit(X_emb)
    _, idx = nn.kneighbors(coords)
    idx = idx.ravel()

    ax.quiver(
        X_emb[idx, 0], X_emb[idx, 1],
        V_pred[idx, 0], V_pred[idx, 1],
        angles="xy", scale_units="xy", scale=3,
        width=0.003,
        headwidth=4.5, headlength=4.0, headaxislength=2.3,
        minlength=0.2,
        color=color, alpha=0.9
    )
    return ax


# ===============================================================
# 1) Embedding (spatial coords from FlowMap)
# ===============================================================
X_emb = emb.X_emb                         # spatial coordinates (already set)
labels = np.asarray(adata.obs["celltype_mapped_refined"].values)  # default


# ===============================================================
# 2) Simplified 7-category palette
# ===============================================================
# --- apply simplified labels ---
labels_simplified = labels.copy()
labels_simplified = np.array([simplify_map[l] for l in labels_simplified])

# --- 7-color high-quality palette ---
palette_simple = {
    "Neural": "#4C72B0",               # blue
    "Mesoderm (Somitic)": "#55A868",   # green
    "Mesoderm (LPM/Heart)": "#C44E52", # red
    "Endoderm": "#8172B2",             # purple
    "Endothelium/Blood": "#CCB974",    # tan
    "Surface ectoderm": "#64B5CD",     # cyan
    "Allantois": "#8C8C8C",            # gray
}

# --- turn into color array ---
cell_colors = np.array([palette_simple[l] for l in labels_simplified])


# ===============================================================
# 3) Grid seeds + TPS prediction
# ===============================================================
# compute grid on X_emb coordinates
Xg, keep_mass, _ = compute_velocity_on_grid(X_emb, grid_size=20, min_mass=0.01)

# mask seeds outside cell manifold
keep_inside = points_inside_mask(X_emb, Xg, k=8, radius_scale=1.2)
Xg = Xg[keep_inside]

# TPS prediction
Vg = emb.tps_vf.predict(Xg)

# full-cell predictions (for manual arrows)
V_cells = emb.tps_vf.predict(X_emb)


# ===============================================================
# 4) Plot
# ===============================================================
fig, ax = plt.subplots(figsize=(10, 10))

# scatter
ax.scatter(
    X_emb[:, 0], X_emb[:, 1],
    c=cell_colors, s=24, alpha=0.3, linewidths=0
)

# quiver (grid arrows)
ax.quiver(
    Xg[:, 0], Xg[:, 1],
    Vg[:, 0], Vg[:, 1],
    angles="xy", scale_units="xy", scale=150,
    width=0.005,             # thicker shaft (was 0.003)
    headwidth=5.0,           # bigger head
    headlength=4.0,
    headaxislength=3.5,
    minlength=0.2,
    color="k", alpha=0.95     # slightly darker also looks better
)

# optional manual arrows — EMPTY for now
patch_coords = []     # <<< placeholder
add_manual_arrows(ax, patch_coords, X_emb, V_cells)

ax.invert_yaxis()
# clean axes
ax.set_aspect("equal")
ax.set_xticks([]); ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
# ===============================================================
# Legend for simplified spatial layers — transparent, no box
# ===============================================================

uniq = [
    "Neural",
    "Mesoderm (Somitic)",
    "Mesoderm (LPM/Heart)",
    "Endoderm",
    "Endothelium/Blood",
    "Surface ectoderm",
    "Allantois",
]

# create handles
handles = [
    plt.Line2D([], [], marker='o', linestyle='',
               color=palette_simple[lab],
               label=lab, markersize=10)
    for lab in uniq
]

# transparent figure
fig, ax = plt.subplots(figsize=(3, 2))

# draw legend with no frame
leg = ax.legend(
    handles=handles,
    fontsize=10,
    loc='center',
    frameon=False      # <-- removes box!
)

ax.axis("off")

# save with transparent background
plt.tight_layout()

plt.show()


In [ ]:
from scripts.plotting import *
from scripts.FieldReconstructionEvaluator import *

emb.fit_gene_level_splines(dof_gene=50, dof_vf_gene=50)
evaluator = FieldReconstructionEvaluator(emb)
res = evaluator.evaluate_gene_fit()

In [ ]:
%load_ext autoreload
%autoreload 2
from scripts.plotting import *

emb.gene_names = adata.var_names
x = np.array(res['expr_corr_gene_r2'])
y = np.array(res['vel_corr_gene_r2'])

plt.figure(figsize=(6,5))
plt.scatter(x, y, alpha=0.6, s=25, color='teal', edgecolor='none')
plt.xlabel('Expression $R^2$')
plt.ylabel('Velocity $R^2$')
plt.title('Expression vs Velocity $R^2$')
plt.plot([0, 1], [0, 1], '--', color='gray', lw=1)  # reference diagonal
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

plot_gene_r2_scatter(res, 
                     emb.gene_names,
                     thr_expr=0.6,
                     thr_vel=0.3)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# genes to plot
genes = ["Ttn", "Lhx2", "Six3", "Hoxd4", "Pax8"]

# extract spatial coords
X = adata.obsm["X_xy_loc"]
Xx, Xy = X[:, 0], X[:, 1]

# 1×5 grid
fig, axes = plt.subplots(1, 5, figsize=(24, 5))

for ax, gene in zip(axes, genes):
    if gene not in adata.var_names:
        print(f"Warning: {gene} not found in var_names!")
        continue
    
    expr = adata[:, gene].X
    # convert sparse → dense if needed
    if hasattr(expr, "toarray"):
        expr = expr.toarray().ravel()
    else:
        expr = np.array(expr).ravel()
    
    sc = ax.scatter(
        Xx, Xy,
        c=expr,
        cmap="viridis",
        s=6,
        alpha=0.85,
        linewidths=0
    )

    ax.set_title(gene, fontsize=14)
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])

    # invert y-axis to match embryo orientation
    ax.invert_yaxis()

    # remove frame
    for spine in ax.spines.values():
        spine.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
from scripts.VectorFieldEmbedder import *

umap_params = {
    "min_dist": 0.7
}

emb_ge = VectorFieldEmbedder(
    X_raw, V_raw,
    dist_method="phase",
    method="umap",
    embed_kwargs=umap_params,
    dof=50,
    alpha=0.75,
    knn_k=30,
    max_tps_points=4000
)


emb_ge.initialize_embedding(seed=42)

In [ ]:
plot_velocity_streamplot(emb_ge.X_emb, emb_ge.tps_vf, 
                         scatter_color=adata.obs["celltype_mapped_refined"].values,
                         cmap="tab20",
                         stream_density=1.5, show_axes=False)

In [ ]:
emb_ge.fit_gene_level_splines(dof_gene=50, dof_vf_gene=50)
evaluator = FieldReconstructionEvaluator(emb_ge)
res = evaluator.evaluate_gene_fit()

In [ ]:
emb.gene_names = adata.var_names
x = np.array(res['expr_corr_gene_r2'])
y = np.array(res['vel_corr_gene_r2'])

plt.figure(figsize=(6,5))
plt.scatter(x, y, alpha=0.6, s=25, color='teal', edgecolor='none')
plt.xlabel('Expression $R^2$')
plt.ylabel('Velocity $R^2$')
plt.title('Expression vs Velocity $R^2$')
plt.plot([0, 1], [0, 1], '--', color='gray', lw=1)  # reference diagonal
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

plot_gene_r2_scatter(res, 
                     emb.gene_names,
                     thr_expr=0.6,
                     thr_vel=0.3)